In [2]:
# ==========================================================
# TASK 8
# Semantic Segmentation using a Small U-Net
# ==========================================================

# ----------------------------------------------------------
# Step 1 - Import Libraries
# ----------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F

print("Libraries Imported Successfully")


# ----------------------------------------------------------
# Step 2 - Create Sample Data
# ----------------------------------------------------------

torch.manual_seed(42)

images = torch.randn(
    8,
    1,
    64,
    64
)

masks = torch.randint(
    0,
    2,
    (8, 1, 64, 64)
).float()

print("Image Shape:", images.shape)
print("Mask Shape :", masks.shape)


# ----------------------------------------------------------
# Step 3 - Create U-Net
# ----------------------------------------------------------

class SmallUNet(nn.Module):

    def __init__(self):

        super(SmallUNet, self).__init__()

        # Encoder 1
        self.encoder1 = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1),
            nn.ReLU()
        )

        # Encoder 2
        self.encoder2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU()
        )

        # Pooling
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU()
        )

        # Upsampling
        self.up = nn.ConvTranspose2d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        # Decoder
        # 32 from upsampling + 32 from skip connection = 64
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1),
            nn.ReLU()
        )

        # Final output
        self.output_layer = nn.Conv2d(
            16,
            1,
            kernel_size=1
        )


    def forward(self, x):

        # Encoder
        first = self.encoder1(x)

        pooled1 = self.pool(first)

        second = self.encoder2(pooled1)

        pooled2 = self.pool(second)

        # Bottleneck
        middle = self.bottleneck(pooled2)

        # Decoder
        upsampled = self.up(middle)

        # Skip connection
        combined = torch.cat(
            [upsampled, second],
            dim=1
        )

        decoded = self.decoder(combined)

        # Restore original size
        decoded = F.interpolate(
            decoded,
            size=first.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        # Final segmentation mask
        output = self.output_layer(decoded)

        return output


# ----------------------------------------------------------
# Step 4 - Create Model
# ----------------------------------------------------------

model = SmallUNet()

print("U-Net Model Created")


# ----------------------------------------------------------
# Step 5 - Check Model Output
# ----------------------------------------------------------

test_output = model(images)

print("Output Shape:", test_output.shape)


# ----------------------------------------------------------
# Step 6 - Dice Loss
# ----------------------------------------------------------

def dice_loss(prediction, target):

    prediction = torch.sigmoid(prediction)

    smooth = 1e-6

    intersection = torch.sum(
        prediction * target
    )

    total = (
        torch.sum(prediction)
        + torch.sum(target)
    )

    dice_score = (
        2 * intersection + smooth
    ) / (
        total + smooth
    )

    return 1 - dice_score


# ----------------------------------------------------------
# Step 7 - IoU
# ----------------------------------------------------------

def calculate_iou(prediction, target):

    prediction = torch.sigmoid(prediction)

    prediction = (
        prediction > 0.5
    ).float()

    intersection = torch.sum(
        prediction * target
    )

    union = (
        torch.sum(prediction)
        + torch.sum(target)
        - intersection
    )

    iou = (
        intersection + 1e-6
    ) / (
        union + 1e-6
    )

    return iou.item()


# ----------------------------------------------------------
# Step 8 - Create Optimizer
# ----------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ----------------------------------------------------------
# Step 9 - Train Model
# ----------------------------------------------------------

epochs = 5

for epoch in range(epochs):

    optimizer.zero_grad()

    predictions = model(images)

    loss = dice_loss(
        predictions,
        masks
    )

    loss.backward()

    optimizer.step()

    print(
        "Epoch",
        epoch + 1,
        "| Dice Loss =",
        round(loss.item(), 4)
    )


# ----------------------------------------------------------
# Step 10 - Calculate IoU
# ----------------------------------------------------------

with torch.no_grad():

    final_predictions = model(images)

    iou = calculate_iou(
        final_predictions,
        masks
    )


print("\nFinal IoU:", round(iou, 4))


# ----------------------------------------------------------
# Step 11 - Display Results
# ----------------------------------------------------------

print(
    "Prediction Shape:",
    final_predictions.shape
)

print(
    "Target Shape:",
    masks.shape
)


# ----------------------------------------------------------
# Step 12 - Experiment Completed
# ----------------------------------------------------------

print("\nSegmentation Experiment Completed")

Libraries Imported Successfully
Image Shape: torch.Size([8, 1, 64, 64])
Mask Shape : torch.Size([8, 1, 64, 64])
U-Net Model Created
Output Shape: torch.Size([8, 1, 64, 64])
Epoch 1 | Dice Loss = 0.5025
Epoch 2 | Dice Loss = 0.5003
Epoch 3 | Dice Loss = 0.4976
Epoch 4 | Dice Loss = 0.4937
Epoch 5 | Dice Loss = 0.4875

Final IoU: 0.4997
Prediction Shape: torch.Size([8, 1, 64, 64])
Target Shape: torch.Size([8, 1, 64, 64])

Segmentation Experiment Completed
